# EcoTrace — Greenwashing Detection Demo

This notebook walks through the full EcoTrace pipeline on three canonical claims:
1. HIGH RISK — direct contradiction
2. MODERATE RISK — future commitment without current action
3. Failure mode — vocabulary mismatch

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
project_root = Path('.').resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / '.env')

print('Environment ready.')

## 1. Initialize Pipeline

In [ ]:
from src.pipeline import EcoTracePipeline

pipeline = EcoTracePipeline(
    index_path='../models/bi_encoder_index',
    verifier_model='../models/cross_encoder',
)
print('Pipeline initialized.')

## 2. Stage 1 — Retrieval Demo

In [ ]:
claim = 'We have achieved carbon neutrality across all our operations.'
retrieved = pipeline.retriever.retrieve(claim, top_k=5)

print(f'Query: "{claim}"\n')
print('Top-5 Retrieved Evidence:')
for item in retrieved:
    print(f'  [{item["rank"]}] (cos_sim={item["score"]:.4f}) {item["sentence"]}')

## 3. Stage 2 — NLI Verification Demo

In [ ]:
if retrieved:
    evidence = retrieved[0]['sentence']
    nli_result = pipeline.verifier.predict(claim, evidence)
    print(f'Claim:    {claim}')
    print(f'Evidence: {evidence}')
    print(f'Label:    {nli_result["label"]}')
    print(f'P(SUPPORT)={nli_result["probabilities"]["SUPPORT"]:.3f}  '
          f'P(NEUTRAL)={nli_result["probabilities"]["NEUTRAL"]:.3f}  '
          f'P(REFUTE)={nli_result["probabilities"]["REFUTE"]:.3f}')

## 4. Stage 3 — Greenwashing Scoring

In [ ]:
from src.verification.scorer import compute_greenwashing_score, get_verdict

if retrieved:
    probs = nli_result['probabilities']
    score = compute_greenwashing_score(probs, w1=0.7, w2=0.3)
    verdict = get_verdict(score)
    print(f'S_gw = 0.7 × P(Refute) + 0.3 × (1 − P(Support))')
    print(f'     = 0.7 × {probs["REFUTE"]:.3f} + 0.3 × (1 − {probs["SUPPORT"]:.3f})')
    print(f'     = {score:.4f}')
    print(f'Verdict: {verdict}')

## 5. Full Pipeline — All Three Demo Cases

In [ ]:
cases = [
    ('Case 1 — HIGH RISK', 'We have achieved carbon neutrality across all our operations.'),
    ('Case 2 — MODERATE RISK', 'We are committed to reducing emissions by 2050.'),
    ('Case 3 — Failure Mode', 'Our supply chain is 100% deforestation-free.'),
]

for name, claim_text in cases:
    print(f'\n{'='*55}')
    print(f'  {name}')
    print('='*55)
    result = pipeline.analyze(claim_text)
    print(f'CLAIM: "{result["claim"]}"')
    print(f'NLI VERDICT:  {result["nli_results"][0]["label"] if result["nli_results"] else "N/A"}')
    print(f'RISK SCORE:   {result["greenwashing_score"]:.2f}')
    print(f'VERDICT:      {result["verdict"]}')
    print(f'REASONING:    {result["reasoning"]}')
    print(f'Time: {result["processing_time_ms"]:.0f}ms')

## 6. Evaluation Metrics Visualization

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

report_path = Path('../results/evaluation_report.json')
if report_path.exists():
    with open(report_path) as f:
        metrics = json.load(f)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Precision / Recall / F1 bar
    ax = axes[0]
    metric_names = ['Precision', 'Recall', 'F1']
    values = [metrics.get('precision', 0), metrics.get('recall', 0), metrics.get('f1', 0)]
    bars = ax.bar(metric_names, values, color=['#22c55e', '#f59e0b', '#3b82f6'])
    ax.set_ylim(0, 1)
    ax.set_title('Classification Metrics (Macro)', color='white')
    ax.set_facecolor('#0d1117')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', ha='center', color='white')

    # Recall@K
    ax2 = axes[1]
    k_vals = [1, 3, 5]
    recall_vals = [metrics.get(f'recall@{k}', 0) for k in k_vals]
    ax2.plot(k_vals, recall_vals, 'o-', color='#22c55e', linewidth=2, markersize=8)
    ax2.set_xlabel('K')
    ax2.set_ylabel('Recall@K')
    ax2.set_title('Retrieval Recall@K', color='white')
    ax2.set_facecolor('#0d1117')
    ax2.set_ylim(0, 1)

    for ax in axes:
        ax.tick_params(colors='white')
        ax.spines[:].set_color('#374151')

    fig.patch.set_facecolor('#0d1117')
    plt.tight_layout()
    plt.savefig('../results/metrics_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Run scripts/evaluate.py first to generate evaluation_report.json')

## 7. Confusion Matrix

In [ ]:
import seaborn as sns

if report_path.exists() and 'confusion_matrix' in metrics:
    cm = np.array(metrics['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Greens',
        xticklabels=['SUPPORT', 'REFUTE', 'NEUTRAL'],
        yticklabels=['SUPPORT', 'REFUTE', 'NEUTRAL'],
        ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix — EcoTrace NLI')
    plt.tight_layout()
    plt.savefig('../results/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()